In [ ]:
#for stamp
import shutil

# Define source and destination
source_folder = "/kaggle/input/convolve/images" # Path from Step 1
destination_folder = "/kaggle/working/images"

# Copy the folder to the output directory
if not os.path.exists(destination_folder):
    shutil.copytree(source_folder, destination_folder)
    print(f"Folder copied to: {destination_folder}")
else:
    print("Folder already exists in output.")


In [ ]:
#for stamp
import shutil

# Define source and destination
source_folder = "/kaggle/input/labels" # Path from Step 1
destination_folder = "/kaggle/working/labels"

# Copy the folder to the output directory
if not os.path.exists(destination_folder):
    shutil.copytree(source_folder, destination_folder)
    print(f"Folder copied to: {destination_folder}")
else:
    print("Folder already exists in output.")

In [ ]:
# -------- CONFIG --------
DATASET_ROOT = "/kaggle/working"
CLASS_NAMES = ["stamp"]

# -------- CREATE data.yaml --------
yaml_content = f"""
path: {DATASET_ROOT}

train: images/train
val: images/val

nc: {len(CLASS_NAMES)}
names:
"""

for cls in CLASS_NAMES:
    yaml_content += f"  - {cls}\n"

# -------- SAVE FILE --------
with open("data.yaml", "w") as f:
    f.write(yaml_content.strip())

print("✅ data.yaml created successfully!")
print("📄 Saved at: /kaggle/working/data.yaml")


In [ ]:
!pip install -q ultralytics


In [ ]:
import os
import shutil

# Path to the run directory you want to delete
run_dir = '/kaggle/working/runs'

# Specific weight files to delete
files_to_delete = [
    '/kaggle/working/yolov8n.pt',
    '/kaggle/working/yolo26n.pt'
]

# Delete the directory and all its contents
if os.path.exists(run_dir):
    shutil.rmtree(run_dir)
    print(f"Deleted directory: {run_dir}")

# Delete the specific .pt files
for file_path in files_to_delete:
    if os.path.exists(file_path):
        os.remove(file_path)
        print(f"Deleted file: {file_path}")
    else:
        print(f"File not found: {file_path}")


In [ ]:
!find /kaggle/working -name "*.txt" | head -n 20


In [ ]:
# Create the correct YOLO label folders
!mkdir -p labels/train
!mkdir -p labels/val

# Move all label files to labels/train
!mv labels/labels/*.txt labels/train/


In [ ]:
!ls labels/train | head


In [ ]:
!sed -n '1p' labels/train/*.txt | head


In [ ]:
!mkdir -p images/val
!mkdir -p labels/val


In [ ]:
!ls labels/train | shuf | head -n 20 | while read f; do \
  base="${f%.txt}"; \
  img=$(ls images/train/${base}.* 2>/dev/null | head -n 1); \
  if [ -f "$img" ]; then \
    mv "labels/train/$f" labels/val/; \
    mv "$img" images/val/; \
  fi; \
done


In [ ]:
!ls labels/val | wc -l
!ls images/val | wc -l


In [ ]:
import os
import shutil

val_img_dir = "images/val"
val_lbl_dir = "labels/val"
train_img_dir = "images/train"

# Get label base names
label_bases = set(
    os.path.splitext(f)[0]
    for f in os.listdir(val_lbl_dir)
    if f.endswith(".txt")
)

moved = 0

for img in os.listdir(val_img_dir):
    base = os.path.splitext(img)[0]
    if base not in label_bases:
        shutil.move(
            os.path.join(val_img_dir, img),
            os.path.join(train_img_dir, img)
        )
        moved += 1

print(f"✅ Moved {moved} extra images back to images/train")


In [ ]:
from ultralytics import YOLO
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

model = YOLO("yolov8n.pt")

model.train(
    data="data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    workers=4,
    name="stamp_detector_final"
)


In [ ]:
from ultralytics import YOLO

model = YOLO("/kaggle/working/runs/detect/stamp_detector_final/weights/best.pt")

results = model.predict(
    source="/kaggle/working/images/val",
    conf=0.25,
    save=False
)

for r in results:
    if r.boxes is None:
        print("No detections")
        continue

    boxes = r.boxes.xyxy.cpu().numpy()
    scores = r.boxes.conf.cpu().numpy()
    classes = r.boxes.cls.cpu().numpy()

    for box, score, cls in zip(boxes, scores, classes):
        x1, y1, x2, y2 = box
        class_name = r.names[int(cls)]
        print(f"{class_name} | conf={score:.2f} | box=({int(x1)}, {int(y1)}, {int(x2)}, {int(y2)})")


In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
import cv2

# Load trained model
model = YOLO("/kaggle/working/runs/detect/stamp_detector_final/weights/best.pt")

# Run prediction (no saving)
results = model.predict(
    source="/kaggle/working/images/train",
    conf=0.3,
    iou=0.4,
    save=False
)

# Visualize results
for r in results:
    img = r.orig_img.copy()

    if r.boxes is not None and len(r.boxes) > 0:
        boxes = r.boxes.xyxy.cpu().numpy()
        scores = r.boxes.conf.cpu().numpy()

        for box, score in zip(boxes, scores):
            x1, y1, x2, y2 = map(int, box)

            # Draw bounding box
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 3)

            # Draw label
            cv2.putText(
                img,
                f"stamp {score:.2f}",
                (x1, max(y1 - 10, 30)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.9,
                (0, 255, 0),
                2
            )

    # Convert BGR → RGB for matplotlib
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(6, 10))
    plt.imshow(img_rgb)
    plt.axis("off")
    plt.show()
